# Solutions · 15 · Pipe flow of non-Newtonian fluids

Worked solutions to the exercises of [notebook 15](../notebooks/15_pipe_flow_of_non_newtonian_fluids.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engrheo is missing): install it from GitHub
    import engrheo
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engrheo"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engrheo import datasets, fitting, flows
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

In [2]:
conc = pd.read_csv(datasets.path("concrete_flow_curve.csv"), comment="#")
b = fitting.fit_flow_curve(conc.shear_rate_1_s, conc.shear_stress_Pa, "bingham").params

## Exercise 1

How does the concrete pump pressure change if the line diameter is 100, 125 or 150 mm at 30 m³/h? Which part of the pressure (yield stress or viscous) depends most on the diameter?

In [3]:
Q, L = 30 / 3600, 100.0
for D in (0.100, 0.125, 0.150):
    s = flows.pipe_hb(b["tau_y"], b["mu_p"], 1.0, D / 2, Q=Q)
    total, yield_part = s["dp_per_L"] * L, 4 * b["tau_y"] * L / D
    print(f"D = {D*1000:.0f} mm: {total/1e5:6.1f} bar, of which yield stress {yield_part/1e5:5.1f} bar, viscous {(total - yield_part)/1e5:6.1f} bar")

D = 100 mm:  179.6 bar, of which yield stress  25.0 bar, viscous  154.6 bar
D = 125 mm:   86.5 bar, of which yield stress  20.0 bar, viscous   66.5 bar
D = 150 mm:   50.9 bar, of which yield stress  16.6 bar, viscous   34.3 bar


The viscous part falls steeply with diameter (for a fixed flow rate roughly as $1/D^4$), the yield-stress part only
as $1/D$. Larger lines cut pumping pressure dramatically - at the cost of more concrete in the line and heavier
pipework.

## Exercise 2

For the drilling mud, double the flow rate from 250 to 500 gal/min. By what factor does the pressure loss grow, and how does that compare with the laminar power-law scaling $\Delta P \propto Q^n$?

In [4]:
from engrheo import geometry
mud = pd.read_csv(datasets.path("drilling_mud_fann.csv"), comment="#")
tau_m, rate_m = geometry.fann35(mud.rpm.to_numpy(), mud.dial_reading_deg.to_numpy())
fast = rate_m >= 170; o = np.argsort(rate_m[fast])
pl = fitting.fit_flow_curve(rate_m[fast][o], tau_m[fast][o], "power_law").params
r = [flows.pressure_drop_power_law(q * 6.309e-5, 0.1086, 1000.0, 1200.0, pl["K"], pl["n"]) for q in (250, 500)]
print(f"regimes: {r[0]['regime']}, {r[1]['regime']}; pressure ratio {r[1]['dp']/r[0]['dp']:.2f}; laminar power-law scaling 2^n = {2**pl['n']:.2f}")

regimes: laminar, turbulent; pressure ratio 2.83; laminar power-law scaling 2^n = 1.38


Between 250 gal/min (laminar) and 500 gal/min (turbulent) the flow changes regime, and the pressure loss rises
2.8-fold instead of the 1.4-fold that the laminar power-law scaling $Q^n$ would suggest. In turbulent flow the loss
grows roughly as $Q^{1.75}$-$Q^2$. Pump sizing must check the regime and use the matching friction law.

## Exercise 3

**Implement it yourself:** write the Metzner-Reed Reynolds number from its definition and check, for the power-law fluid above, that $f = 16/Re_{MR}$ reproduces the exact laminar pressure gradient from `flows.pipe_power_law`.

In [5]:
R, Q, K, n, rho = 0.025, 1.5e-3, 0.5, 0.4, 1000.0
V, D = Q / (np.pi * R**2), 2 * R
Re = rho * V ** (2 - n) * D**n / (8 ** (n - 1) * K * ((3 * n + 1) / (4 * n)) ** n)
dp_f = 4 * (16 / Re) / D * rho * V**2 / 2
print(f"Re_MR by hand {Re:.3f} (library {flows.reynolds_metzner_reed(rho, V, D, K, n):.3f}); "
      f"f = 16/Re gives {dp_f:.3f} Pa/m, exact laminar solution {flows.pipe_power_law(K, n, R, Q=Q)['dp_per_L']:.3f} Pa/m")

Re_MR by hand 1202.400 (library 1202.400); f = 16/Re gives 310.637 Pa/m, exact laminar solution 310.637 Pa/m


The Metzner-Reed Reynolds number is *defined* so that the laminar friction factor keeps its Newtonian form
16/Re for power-law fluids - which is why the two pressure gradients agree exactly.